<a href="https://colab.research.google.com/github/clobos/Biologia_2026/blob/main/teste_Hipoteses_2_parametros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from scipy import stats

url = ("https://vincentarelbundock.github.io"
       "/Rdatasets/csv/palmerpenguins/penguins.csv")
penguins = pd.read_csv(url)

# Pinguins Adelie da ilha Torgersen, sem valores ausentes
adelie = penguins[(penguins["species"] == "Adelie") & (penguins["island"] == "Torgersen")]
adelie = adelie.dropna(subset=["body_mass_g", "sex"])

# Massa corporal (g) de cada grupo
machos = adelie.loc[adelie["sex"] == "male", "body_mass_g"]
femeas = adelie.loc[adelie["sex"] == "female", "body_mass_g"]

print(f"número de machos:",len(machos))
print(machos.head())


número de machos: 23
0     3750.0
5     3650.0
7     4675.0
13    3800.0
14    4400.0
Name: body_mass_g, dtype: float64


In [2]:
print(f"número de fêmeas:",len(femeas))
print(femeas.head())


número de fêmeas: 24
1     3800.0
2     3250.0
4     3450.0
6     3625.0
12    3200.0
Name: body_mass_g, dtype: float64


# Teste t para Duas Médias Independentes: variâncias iguais

In [3]:
# H0: mu_macho = mu_femea
res = stats.ttest_ind(machos, femeas, equal_var=True, alternative="two-sided")
print(f"T = {res.statistic:.3f}")
print(f"p-valor = {res.pvalue:.4f}")
print(f"gl = {len(machos) + len(femeas) - 2}")

T = 6.851
p-valor = 0.0000
gl = 45


In [4]:
print(res)

TtestResult(statistic=np.float64(6.851212911020037), pvalue=np.float64(1.69419702278338e-08), df=np.float64(45.0))


# Teste t para Duas Médias Independentes: variâncias diferentes

In [ ]:
# H0: mu_macho = mu_femea
res = stats.ttest_ind(machos, femeas, equal_var=False, alternative="two-sided")
print(f"T = {res.statistic:.3f}")
print(f"p-valor = {res.pvalue:.4f}")
print(f"gl = {res.df:.2f}")

T = 6.800
p-valor = 0.0000
gl = 39.10


# Teste t pareado

In [5]:
import pandas as pd
from scipy import stats

url = ("https://vincentarelbundock.github.io"
       "/Rdatasets/csv/datasets/sleep.csv")
sleep = pd.read_csv(url)

sleep

,rownames,extra,group,ID
0,1,0.7,1,1
1,2,-1.6,1,2
2,3,-0.2,1,3
3,4,-1.2,1,4
4,5,-0.1,1,5
5,6,3.4,1,6
6,7,3.7,1,7
7,8,0.8,1,8
8,9,0.0,1,9
9,10,2.0,1,10


In [7]:
# Organizar em formato largo (um par por linha)
wide = sleep.pivot(index="ID", columns="group", values="extra")
wide

group,1,2
ID,,
1,0.7,1.9
2,-1.6,0.8
3,-0.2,1.1
4,-1.2,0.1
5,-0.1,-0.1
6,3.4,4.4
7,3.7,5.5
8,0.8,1.6
9,0.0,4.6


In [8]:
wide.columns = ["trat1", "trat2"]

# Manter somente pares completos
wide = wide.dropna()
wide


,trat1,trat2
ID,,
1,0.7,1.9
2,-1.6,0.8
3,-0.2,1.1
4,-1.2,0.1
5,-0.1,-0.1
6,3.4,4.4
7,3.7,5.5
8,0.8,1.6
9,0.0,4.6


In [10]:
# Teste t pareado
resultado = stats.ttest_rel(wide["trat2"],
                            wide["trat1"],
                            alternative='two-sided')
print(resultado)

TtestResult(statistic=np.float64(4.062127683382037), pvalue=np.float64(0.00283289019738427), df=np.int64(9))


In [9]:
# Teste t pareado
resultado = stats.ttest_rel(wide["trat1"],
                            wide["trat2"],
                            alternative='two-sided')
print(resultado)

TtestResult(statistic=np.float64(-4.062127683382037), pvalue=np.float64(0.00283289019738427), df=np.int64(9))


In [ ]:
# Conferir via teste de uma amostra nas diferenças
diferencas = wide["trat2"] - wide["trat1"]
stats.ttest_1samp(diferencas,
                  popmean=0,
                  alternative='two-sided')

TtestResult(statistic=np.float64(4.062127683382037), pvalue=np.float64(0.00283289019738427), df=np.int64(9))

# Ideias do valor-p

In [ ]:
import numpy as np
from scipy import stats
from scipy.integrate import quad

# Dados simulados (n = 37)
x = np.array([52.97, 55.30, 57.16, 41.33, 51.68, 61.36, 47.91, 51.61, 53.19, 60.36, 46.33, 47.89, 59.59, 51.76, 48.09, 58.12, 41.81, 58.17, 58.42, 47.15, 51.76, 47.63, 52.57, 44.06, 53.18, 57.33, 59.08, 55.88, 42.40, 46.83, 47.23, 45.92, 54.13, 51.48, 42.94, 40.60, 56.11])
mu0 = 50.0
alternativa = 'greater'   # 'two-sided', 'greater' ou 'less'
alpha = 0.05

n = x.size
gl = n - 1
xbar = x.mean()
s = x.std(ddof=1)
ep = s / np.sqrt(n)
t_obs = (xbar - mu0) / ep

# 1) p-valor pela distribuição acumulada e pela sobrevivência
if alternativa == 'two-sided':
    p_cdf = 2 * stats.t.sf(abs(t_obs), gl)
elif alternativa == 'greater':
    p_cdf = stats.t.sf(t_obs, gl)
else:
    p_cdf = stats.t.cdf(t_obs, gl)

# 2) p-valor como integral da densidade t de Student com n - 1 graus de liberdade
f = lambda u: stats.t.pdf(u, gl)
if alternativa == 'two-sided':
    p_int, _ = quad(f, abs(t_obs), np.inf)
    p_int *= 2
elif alternativa == 'greater':
    p_int, _ = quad(f, t_obs, np.inf)
else:
    p_int, _ = quad(f, -np.inf, t_obs)

# 3) Conferência com o teste pronto do SciPy
res = stats.ttest_1samp(x, popmean=mu0, alternative=alternativa)

print(f"t_obs = {t_obs:.4f}   (gl = {gl})")
print(f"p (cdf ou sf)   = {p_cdf:.6f}")
print(f"p (integral)    = {p_int:.6f}")
print(f"p (ttest_1samp) = {res.pvalue:.6f}")
print("Decisão:", "rejeitar H0" if p_int <= alpha else "não rejeitar H0")

t_obs = 1.3666   (gl = 36)
p (cdf ou sf)   = 0.090107
p (integral)    = 0.090107
p (ttest_1samp) = 0.090107
Decisão: não rejeitar H0
